# 02 — Clean and enrich job postings

This notebook reads the Bronze file, cleans text in visible steps, derives fields used in the analysis, and creates the Silver working file.

In [ ]:
# Import libraries, load the settings, and locate the output from notebook 01.
from pathlib import Path
import hashlib
import json
import html
import unicodedata
import re
import pandas as pd
import yaml


def find_project_root(start: Path) -> Path:
    """Find the project root without assuming where Jupyter was launched."""
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "data").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project folder.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
with (PROJECT_ROOT / "notebooks" / "analysis_settings.yaml").open(encoding="utf-8") as file:
    settings = yaml.safe_load(file)

BRONZE_PATH = PROJECT_ROOT / "notebooks" / "data" / "working" / "01_bronze_postings.csv"
SILVER_PATH = PROJECT_ROOT / "notebooks" / "data" / "working" / "02_silver_postings.csv"
if not BRONZE_PATH.exists():
    raise FileNotFoundError("Run notebook 01 before this notebook.")

bronze = pd.read_csv(BRONZE_PATH, low_memory=False, dtype={"source_job_id": "string"})
print(f"Bronze rows loaded: {len(bronze):,}")

In [ ]:
# Define small, readable cleaning functions instead of hiding the logic in one long expression.
def as_text(value) -> str:
    """Return a safe text value; missing values become an empty string."""
    return "" if pd.isna(value) else str(value)


def clean_text(value) -> str:
    """Remove HTML-like tags, normalise whitespace, and keep readable source wording."""
    text = html.unescape(as_text(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalise_label(value) -> str:
    """Create a case-insensitive comparison value while preserving raw fields separately."""
    text = clean_text(value).lower()
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"[^\w]+", " ", text, flags=re.UNICODE).strip()


def derive_seniority(title) -> str:
    """Use complete words; senior and lead take precedence over junior wording."""
    text = normalise_label(title)
    if re.search(r"\b(lead|principal|staff|manager|director|head|vice president)\b", text):
        return "lead"
    if re.search(r"\b(senior|sr)\b", text):
        return "senior"
    if re.search(r"\b(intern|internship|trainee|graduate|fresher|junior|jr|entry level)\b", text):
        return "entry"
    if re.search(r"\b(mid level|intermediate|ii|specialist)\b", text):
        return "mid"
    return "unknown"


In [ ]:
# Parse explicit numeric experience phrases. Unrecognised wording stays blank.
def extract_experience_range(text) -> tuple[float | None, float | None]:
    """Keep numeric experience only when nearby wording refers to an applicant."""
    text = as_text(text).lower().replace("–", "-").replace("—", "-")
    pattern = (
        r"(?<![\w.])(\d+(?:\.\d+)?)\s*(?:(?:-|to)\s*(\d+(?:\.\d+)?))?"
        r"\s*\+?\s*years?\s+(?:of\s+)?"
        r"(?:(?:relevant|professional|work|related|industry)\s+)?experience\b"
    )
    for match in re.finditer(pattern, text):
        # Be conservative: a number alone can describe the employer or manager.
        before = text[max(0, match.start() - 180):match.start()]
        after = text[match.end():match.end() + 100]
        candidate_before = re.search(
            r"\b(require[ds]?|required|requirements|qualifications|minimum|at least|"
            r"must have|you have|you bring|candidate|applicant|possess|"
            r"bachelor|master|degree|diploma|preferred|ideally)\b", before
        )
        candidate_after = re.search(r"\b(required|preferred|desired|mandatory|essential)\b", after)
        employer_context = re.search(
            r"\b(who brings|we have|we bring|we offer|our team|our company|"
            r"company overview|company description|national leader|"
            r"long standing business|management team)\b", before + after
        )
        if not (candidate_before or candidate_after) or employer_context:
            continue
        minimum = float(match.group(1))
        maximum = float(match.group(2)) if match.group(2) else minimum
        if 0 <= minimum <= maximum <= 50:
            return minimum, maximum
    return None, None


def derive_workplace_type(remote_allowed) -> str:
    """1 confirms remote permission; blank or 0 does not prove on-site work."""
    return "remote" if pd.notna(remote_allowed) and float(remote_allowed) == 1 else "unknown"


# Small examples check rules used below. Read why each expected result makes sense.
assert derive_seniority("Internal Auditor") == "unknown"
assert derive_seniority("Senior Graduate Engineer") == "senior"
assert extract_experience_range("Our company has 50 years of history") == (None, None)
assert extract_experience_range("Requires 2–4 years of relevant experience") == (2.0, 4.0)
assert extract_experience_range("5-2 years experience") == (None, None)
assert derive_workplace_type(None) == "unknown"
assert extract_experience_range("Your manager, who brings 20 years of experience in manufacturing") == (None, None)
assert extract_experience_range("We have 25 years of experience across our management team") == (None, None)
assert extract_experience_range("Minimum of 3 years of experience required") == (3.0, 3.0)
assert extract_experience_range("5 years of experience preferred") == (5.0, 5.0)


In [ ]:
# Apply the cleaning rules while preserving original text fields for a later human review.
silver = bronze.copy()
silver["company_normalized"] = silver["company_raw"].map(normalise_label)
silver["title_normalized"] = silver["title_raw"].map(normalise_label)
silver["location_normalized"] = silver["location_raw"].map(normalise_label)
silver["description_clean"] = silver["description_raw"].map(clean_text)
silver["description_word_count"] = silver["description_clean"].str.split().str.len().fillna(0).astype(int)
silver["seniority_title"] = silver["title_raw"].map(derive_seniority)
silver["workplace_type"] = silver["remote_allowed"].map(derive_workplace_type)
silver["employment_type"] = silver["employment_type_raw"].fillna("Unknown")

# Store experience values in two separate numeric columns so later rules can use them directly.
experience_ranges = silver["description_clean"].map(extract_experience_range)
silver[["experience_min_years", "experience_max_years"]] = pd.DataFrame(
    experience_ranges.tolist(), index=silver.index
)
silver["experience_numeric_present"] = silver["experience_min_years"].notna()

In [ ]:
# Convert salary and timestamp fields without inventing a value when the source is blank.
silver["salary_min"] = pd.to_numeric(silver["salary_min_raw"], errors="coerce")
silver["salary_max"] = pd.to_numeric(silver["salary_max_raw"], errors="coerce")
silver["salary_median"] = pd.to_numeric(silver["salary_median_raw"], errors="coerce")
salary_columns = ["salary_min", "salary_max", "salary_median"]
for column in salary_columns:
    silver.loc[silver[column] < 0, column] = float("nan")
invalid_range = silver["salary_min"] > silver["salary_max"]
silver["salary_invalid_range"] = invalid_range
silver.loc[invalid_range, ["salary_min", "salary_max"]] = float("nan")
silver["salary_numeric_present"] = silver[salary_columns].notna().any(axis=1)
silver["listed_at"] = pd.to_datetime(
    pd.to_numeric(silver["listed_time_raw"], errors="coerce"),
    unit="ms",
    utc=True,
    errors="coerce"
)
snapshot_date = pd.Timestamp(settings["source"]["snapshot_date"], tz="UTC")
silver["posting_age_days"] = (snapshot_date - silver["listed_at"].dt.normalize()).dt.days
silver.loc[silver["posting_age_days"] < 0, "posting_age_days"] = pd.NA

# Create an exact duplicate key. It detects identical records, not fuzzy similarity.
key_fields = silver[["company_normalized", "title_normalized", "location_normalized", "description_clean"]].fillna("").astype(str)
silver["exact_duplicate_key"] = key_fields.apply(lambda row: json.dumps(row.tolist(), ensure_ascii=False), axis=1).map(
    lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest()
)
silver["duplicate_cluster_size"] = silver.groupby("exact_duplicate_key")["exact_duplicate_key"].transform("size")
silver["canonical_job_key"] = "job_" + silver["exact_duplicate_key"]
silver[["title_raw", "seniority_title", "experience_min_years", "salary_numeric_present", "duplicate_cluster_size"]].head(10)

In [ ]:
# Save the Silver file for notebook 03 and display a compact quality profile.
SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)
silver.to_csv(SILVER_PATH, index=False)

profile = pd.DataFrame({
    "rows": [len(silver)],
    "experience_numeric_percent": [round(silver["experience_numeric_present"].mean() * 100, 2)],
    "salary_numeric_percent": [round(silver["salary_numeric_present"].mean() * 100, 2)],
    "exact_duplicate_rows": [int((silver["duplicate_cluster_size"] > 1).sum())]
})
print(f"Created {SILVER_PATH.relative_to(PROJECT_ROOT)}")
profile

## Your checkpoint

Write three sentences here: what changed, one result you checked, and one limitation. Explain one code cell in your own words before opening the next notebook.